In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats

BASE_DIR    = Path(r"D:\Shikha\bias_project")
OUTPUT_DIR  = BASE_DIR / "outputs" / "embeddings"
GSS_DIR     = BASE_DIR / "validation_data" / "gss" / "processed"

# Load all three bias score files
w2v   = pd.read_csv(OUTPUT_DIR / "word2vec" / "weat6_scores_word2vec.csv")
glove = pd.read_csv(OUTPUT_DIR / "glove"    / "weat6_scores_glove.csv")
bert  = pd.read_csv(OUTPUT_DIR / "bert"     / "seat_scores_bert.csv")

# Rename score columns for clarity
w2v   = w2v[['state', 'weat6_score']].rename(columns={'weat6_score': 'w2v_score'})
glove = glove[['state', 'weat6_score']].rename(columns={'weat6_score': 'glove_score'})
bert  = bert[['state', 'seat_score']].rename(columns={'seat_score': 'bert_score'})

# Merge all three on state
scores = w2v.merge(glove, on='state').merge(bert, on='state')

# Load GSS region-year data
gss = pd.read_csv(GSS_DIR / "gss_region_year_clean.csv")

print(f"Word2Vec scores : {len(w2v)} states")
print(f"GloVe scores    : {len(glove)} states")
print(f"BERT scores     : {len(bert)} states")
print(f"Merged (all 3)  : {len(scores)} states")
print(f"GSS panel       : {len(gss)} region-year cells")
print(f"\nSample merged scores:")
print(scores.head(10))

Word2Vec scores : 28 states
GloVe scores    : 50 states
BERT scores     : 50 states
Merged (all 3)  : 28 states
GSS panel       : 261 region-year cells

Sample merged scores:
  state  w2v_score  glove_score  bert_score
0    AK    -0.0000       0.1703      0.0389
1    AR    -0.0003       0.1780      0.0155
2    CA    -0.0569       0.2200      0.0196
3    CT    -0.0004       0.1466      0.0191
4    GA     0.0058       0.1636      0.0097
5    HI    -0.0003       0.1649      0.0103
6    IA    -0.0016       0.1638      0.0282
7    IL    -0.0523       0.0933      0.0026
8    IN    -0.0098       0.2216      0.0182
9    KY     0.0016       0.1607      0.0127


In [2]:
# How well do the three methods agree with each other?
# Pearson and Spearman correlations between score columns

print("=== Method Intercorrelation (28 states with all 3 scores) ===\n")

pairs = [
    ('w2v_score', 'glove_score', 'Word2Vec vs GloVe'),
    ('w2v_score', 'bert_score',  'Word2Vec vs BERT'),
    ('glove_score', 'bert_score', 'GloVe vs BERT'),
]

print(f"{'Pair':<22} {'Pearson r':>10} {'p-value':>10} {'Spearman ρ':>12} {'p-value':>10}")
print("-" * 68)

for col1, col2, label in pairs:
    pearson_r, pearson_p   = stats.pearsonr(scores[col1], scores[col2])
    spearman_r, spearman_p = stats.spearmanr(scores[col1], scores[col2])
    print(f"{label:<22} {pearson_r:>10.3f} {pearson_p:>10.3f} {spearman_r:>12.3f} {spearman_p:>10.3f}")

print("""
Interpretation guide:
  r > 0.5  : strong agreement
  r 0.3-0.5: moderate agreement  
  r 0-0.3  : weak agreement
  r < 0    : methods disagree on ranking
p < 0.05 means correlation is statistically significant
""")

=== Method Intercorrelation (28 states with all 3 scores) ===

Pair                    Pearson r    p-value   Spearman ρ    p-value
--------------------------------------------------------------------
Word2Vec vs GloVe           0.032      0.870        0.057      0.773
Word2Vec vs BERT            0.200      0.307        0.143      0.467
GloVe vs BERT               0.028      0.886        0.109      0.581

Interpretation guide:
  r > 0.5  : strong agreement
  r 0.3-0.5: moderate agreement  
  r 0-0.3  : weak agreement
  r < 0    : methods disagree on ranking
p < 0.05 means correlation is statistically significant



In [3]:
# Map states to GSS regions (9-category Census divisions)
STATE_TO_REGION = {
    'CT': 1, 'ME': 1, 'MA': 1, 'NH': 1, 'RI': 1, 'VT': 1,            # New England
    'NJ': 2, 'NY': 2, 'PA': 2,                                        # Middle Atlantic
    'IL': 3, 'IN': 3, 'MI': 3, 'OH': 3, 'WI': 3,                      # East North Central
    'IA': 4, 'KS': 4, 'MN': 4, 'MO': 4, 'NE': 4, 'ND': 4, 'SD': 4,    # West North Central
    'DE': 5, 'FL': 5, 'GA': 5, 'MD': 5, 'NC': 5, 'SC': 5, 'VA': 5, 'WV': 5,  # South Atlantic
    'AL': 6, 'KY': 6, 'MS': 6, 'TN': 6,                               # East South Central
    'AR': 7, 'LA': 7, 'OK': 7, 'TX': 7,                               # West South Central
    'AZ': 8, 'CO': 8, 'ID': 8, 'MT': 8, 'NV': 8, 'NM': 8, 'UT': 8, 'WY': 8,  # Mountain
    'AK': 9, 'CA': 9, 'HI': 9, 'OR': 9, 'WA': 9                       # Pacific
}

# GSS recent years average (2010-2022) per region — most comparable to modern corpus
gss_recent = (
    gss[gss['year'] >= 2010]
    .groupby('region')['composite_score']
    .mean()
    .reset_index()
    .rename(columns={'composite_score': 'gss_attitude'})
)

# Add region to scores, merge GSS
scores['region'] = scores['state'].map(STATE_TO_REGION)
val = scores.merge(gss_recent, on='region')

# Also do this for GloVe and BERT with all 50 states
glove_all = pd.read_csv(OUTPUT_DIR / "glove" / "weat6_scores_glove.csv")[['state','weat6_score']]
bert_all  = pd.read_csv(OUTPUT_DIR / "bert"  / "seat_scores_bert.csv")[['state','seat_score']]
all50 = glove_all.merge(bert_all, on='state')
all50['region'] = all50['state'].map(STATE_TO_REGION)
all50 = all50.merge(gss_recent, on='region')

print("=== GSS External Validation ===")
print("(GSS composite: HIGHER = LESS traditional attitudes)\n")

print("--- 28-state sample (all 3 methods) ---")
for col, label in [('w2v_score','Word2Vec'), ('glove_score','GloVe'), ('bert_score','BERT')]:
    r, p = stats.pearsonr(val[col], val['gss_attitude'])
    print(f"{label:<10} vs GSS: r = {r:>7.3f}, p = {p:.3f}")

print("\n--- 50-state sample (GloVe & BERT only) ---")
for col, label in [('weat6_score','GloVe'), ('seat_score','BERT')]:
    r, p = stats.pearsonr(all50[col], all50['gss_attitude'])
    print(f"{label:<10} vs GSS: r = {r:>7.3f}, p = {p:.3f}")

print("""
Expected direction: NEGATIVE correlation
(higher language bias should pair with lower GSS score = more traditional attitudes)
""")

=== GSS External Validation ===
(GSS composite: HIGHER = LESS traditional attitudes)

--- 28-state sample (all 3 methods) ---
Word2Vec   vs GSS: r =  -0.175, p = 0.374
GloVe      vs GSS: r =  -0.097, p = 0.623
BERT       vs GSS: r =   0.089, p = 0.651

--- 50-state sample (GloVe & BERT only) ---
GloVe      vs GSS: r =   0.017, p = 0.907
BERT       vs GSS: r =  -0.015, p = 0.916

Expected direction: NEGATIVE correlation
(higher language bias should pair with lower GSS score = more traditional attitudes)

